In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm
import re

## Comments Dataset

In [3]:
comments = pd.read_csv(r'C:\Users\New Owner\OneDrive\Documents\April DS Code Jam\youtube_sentiment_analysis\datasets\archive\UScomments.csv', on_bad_lines='skip', encoding= 'utf-8', low_memory=False) # Load the dataset
display(comments.head()) # Display the first few rows of the dataset
display(comments.info()) # Display the information about the dataset

,video_id,comment_text,likes,replies
0,XpVt6Z1Gjjo,Logan Paul it's yo big day ‼️‼️‼️,4,0
1,XpVt6Z1Gjjo,I've been following you from the start of your...,3,0
2,XpVt6Z1Gjjo,Say hi to Kong and maverick for me,3,0
3,XpVt6Z1Gjjo,MY FAN . attendance,3,0
4,XpVt6Z1Gjjo,trending 😉,3,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691400 entries, 0 to 691399
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   video_id      691400 non-null  object
 1   comment_text  691374 non-null  object
 2   likes         691400 non-null  object
 3   replies       691400 non-null  object
dtypes: object(4)
memory usage: 21.1+ MB


None

In [4]:
comments.loc[comments['comment_text'].isna()]  # Filter rows based on condition

,video_id,comment_text,likes,replies
1417,1L7JFN7tQLs,NaN,0,0
76134,7YAAyUFL1GQ,NaN,0,0
215218,KUCHBBCj77I,NaN,0,0
234226,KUCHBBCj77I,NaN,0,0
306019,s3Hk_lDw5yo,NaN,0,0
332811,zrOHeEA14kQ,NaN,0,0
357506,zmg9tVaMVd4,NaN,0,0
379582,zmg9tVaMVd4,NaN,0,0
403013,9eea7_7OBZQ,NaN,0,0
425238,6l5P7jHUcjI,NaN,0,0


In [5]:
comments.dropna(inplace=True) # Drop rows with null values

In [6]:
comments.info() # Check the data types and null values

<class 'pandas.core.frame.DataFrame'>
Index: 691374 entries, 0 to 691399
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   video_id      691374 non-null  object
 1   comment_text  691374 non-null  object
 2   likes         691374 non-null  object
 3   replies       691374 non-null  object
dtypes: object(4)
memory usage: 26.4+ MB


## Videos Dataset

In [7]:
videos = pd.read_csv(r'C:\Users\New Owner\OneDrive\Documents\April DS Code Jam\youtube_sentiment_analysis\datasets\archive\USvideos.csv', on_bad_lines='skip', encoding= 'utf-8') # Display the first 10 rows of the dataset
display(videos) # Display the videos dataset
display(videos.info()) # Display the data types and null values of the videos dataset

,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,date
0,XpVt6Z1Gjjo,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,24,logan paul vlog|logan paul|logan|paul|olympics...,4394029,320053,5931,46245,https://i.ytimg.com/vi/XpVt6Z1Gjjo/default.jpg,13.09
1,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,7860119,185853,26679,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,13.09
2,cLdxuaxaQwc,My Response,PewDiePie,22,[none],5845909,576597,39774,170708,https://i.ytimg.com/vi/cLdxuaxaQwc/default.jpg,13.09
3,WYYvHb03Eog,Apple iPhone X first look,The Verge,28,apple iphone x hands on|Apple iPhone X|iPhone ...,2642103,24975,4542,12829,https://i.ytimg.com/vi/WYYvHb03Eog/default.jpg,13.09
4,sjlHnJvXdQs,iPhone X (parody),jacksfilms,23,jacksfilms|parody|parodies|iphone|iphone x|iph...,1168130,96666,568,6666,https://i.ytimg.com/vi/sjlHnJvXdQs/default.jpg,13.09
...,...,...,...,...,...,...,...,...,...,...,...
7987,xlu6i6lT_vk,How Do MASSIVE Sinkholes Form?,Life Noggin,27,sinkhole|how do sinkholes form|sinkhole in wat...,440393,14362,390,1575,https://i.ytimg.com/vi/xlu6i6lT_vk/default.jpg,22.10
7988,qRoVlH1OcI4,Trump slams Clinton for defending NFL anthem p...,Business Insider,25,Business Insider|Donald Trump|Hillary Clinton|...,55762,1265,760,1873,https://i.ytimg.com/vi/qRoVlH1OcI4/default.jpg,22.10
7989,EoejGgUNmVU,LP - Lost On You (A Night at The McKittrick Ho...,LP,10,LP|Death Valley|Other People|Lost On You|The M...,142908,7088,68,437,https://i.ytimg.com/vi/EoejGgUNmVU/default.jpg,22.10
7990,MT1CMTI0EVw,Tré Melvin @ #YouTubeBlack FanFest Washington ...,YouTube FanFest,24,YouTube FanFest|#YTFF|Washington DC|USA|YTFF|#...,24532,2148,77,0,https://i.ytimg.com/vi/MT1CMTI0EVw/default.jpg,22.10


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7992 entries, 0 to 7991
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   video_id        7992 non-null   object 
 1   title           7992 non-null   object 
 2   channel_title   7992 non-null   object 
 3   category_id     7992 non-null   int64  
 4   tags            7992 non-null   object 
 5   views           7992 non-null   int64  
 6   likes           7992 non-null   int64  
 7   dislikes        7992 non-null   int64  
 8   comment_total   7992 non-null   int64  
 9   thumbnail_link  7992 non-null   object 
 10  date            7992 non-null   float64
dtypes: float64(1), int64(5), object(5)
memory usage: 686.9+ KB


None

Note: understand the relationship between the two datasets,
the comments datasets contain all of the comments for a certain video(marked with id)

## Calculating Like-to-Dislike Ratio

In [8]:
videos['l_d_ratio'] = videos['likes'] / videos['dislikes'] # Calculate the like-dislike ratio
videos = videos.drop(columns=['date']) # Drop the date column
videos.head(10)

,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,l_d_ratio
0,XpVt6Z1Gjjo,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,24,logan paul vlog|logan paul|logan|paul|olympics...,4394029,320053,5931,46245,https://i.ytimg.com/vi/XpVt6Z1Gjjo/default.jpg,53.962738
1,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,7860119,185853,26679,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,6.966266
2,cLdxuaxaQwc,My Response,PewDiePie,22,[none],5845909,576597,39774,170708,https://i.ytimg.com/vi/cLdxuaxaQwc/default.jpg,14.496832
3,WYYvHb03Eog,Apple iPhone X first look,The Verge,28,apple iphone x hands on|Apple iPhone X|iPhone ...,2642103,24975,4542,12829,https://i.ytimg.com/vi/WYYvHb03Eog/default.jpg,5.498679
4,sjlHnJvXdQs,iPhone X (parody),jacksfilms,23,jacksfilms|parody|parodies|iphone|iphone x|iph...,1168130,96666,568,6666,https://i.ytimg.com/vi/sjlHnJvXdQs/default.jpg,170.186620
5,cMKX2tE5Luk,The Disaster Artist | Official Trailer HD | A24,A24,1,a24|a24 films|a24 trailers|independent films|t...,1311445,34507,544,3040,https://i.ytimg.com/vi/cMKX2tE5Luk/default.jpg,63.431985
6,8wNr-NQImFg,"The Check In: HUD, Ben Carson and Hurricanes",Late Night with Seth Meyers,23,Late night|Seth Meyers|check in|hud|Ben Carson...,666169,9985,297,1071,https://i.ytimg.com/vi/8wNr-NQImFg/default.jpg,33.619529
7,_HTXMhKWqnA,iPhone X Impressions & Hands On!,Marques Brownlee,28,iPhone X|iphone x|iphone 10|iPhone X impressio...,1728614,74062,2180,15297,https://i.ytimg.com/vi/_HTXMhKWqnA/default.jpg,33.973394
8,_ANP3HR1jsM,ATTACKED BY A POLICE DOG!!,RomanAtwoodVlogs,22,Roman Atwood|Roman|Atwood|roman atwood vlogs|f...,1338533,69687,678,5643,https://i.ytimg.com/vi/_ANP3HR1jsM/default.jpg,102.783186
9,zgLtEob6X-Q,Honest Trailers - The Mummy (2017),Screen Junkies,1,screenjunkies|screen junkies|screenjunkies new...,1056891,29943,878,4046,https://i.ytimg.com/vi/zgLtEob6X-Q/default.jpg,34.103645


## Text Cleaning

In [9]:
def clear_text(text):
    text = text.lower() # Convert to lowercase
    pattern = r'[^a-zA-Z\s]' # Regular expression pattern to match special characters and punctuation 
    text = re.sub(pattern, " ", text) # Remove special characters, including punctuation
    return text

In [10]:
comments['comment_text'] = comments['comment_text'].astype(str).apply(clear_text)
comments['comment_text'].head(10)

0                    logan paul it s yo big day       
1    i ve been following you from the start of your...
2                   say hi to kong and maverick for me
3                                  my fan   attendance
4                                           trending  
5                                 on trending ayyeeeee
6                                 the end though      
7                                    trending         
8                          happy one year vlogaversary
9    you and your shit brother may have single hand...
Name: comment_text, dtype: object

## Tokenization/ Stop Word Removal / Lemmetization

In [11]:
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')


stop_words = set(stopwords.words('english')) # Set of English stop words
lemmatizer = WordNetLemmatizer() # Initialize the lemmatizer

comments['tokenized_text'] = comments['comment_text'].fillna("").astype(str).apply(word_tokenize) # Apply tokenization to the 'comment_text' column
comments['tokenized_text'] = comments['tokenized_text'].apply(lambda tokens: [word for word in tokens if word not in stop_words]) # Remove stop words from the tokenized text
comments['tokenized_text'] = comments['tokenized_text'].apply(lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]) # Apply lemmatization to the tokenized text

[nltk_data] Downloading package punkt_tab to C:\Users\New
[nltk_data]     Owner\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\New
[nltk_data]     Owner\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\New
[nltk_data]     Owner\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [12]:
# Join the tokenized words back into a single string and save it back to the column
comments['tokenized_text'] = comments['tokenized_text'].apply(lambda tokens: ' '.join(tokens))

In [13]:
comments.head(10)

,video_id,comment_text,likes,replies,tokenized_text
0,XpVt6Z1Gjjo,logan paul it s yo big day,4,0,logan paul yo big day
1,XpVt6Z1Gjjo,i ve been following you from the start of your...,3,0,following start vine channel seen vlogs
2,XpVt6Z1Gjjo,say hi to kong and maverick for me,3,0,say hi kong maverick
3,XpVt6Z1Gjjo,my fan attendance,3,0,fan attendance
4,XpVt6Z1Gjjo,trending,3,0,trending
5,XpVt6Z1Gjjo,on trending ayyeeeee,3,0,trending ayyeeeee
6,XpVt6Z1Gjjo,the end though,4,0,end though
7,XpVt6Z1Gjjo,trending,3,0,trending
8,XpVt6Z1Gjjo,happy one year vlogaversary,3,0,happy one year vlogaversary
9,XpVt6Z1Gjjo,you and your shit brother may have single hand...,0,0,shit brother may single handedly ruined youtub...


In [14]:
videos.head(10)

,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,l_d_ratio
0,XpVt6Z1Gjjo,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,24,logan paul vlog|logan paul|logan|paul|olympics...,4394029,320053,5931,46245,https://i.ytimg.com/vi/XpVt6Z1Gjjo/default.jpg,53.962738
1,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,7860119,185853,26679,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,6.966266
2,cLdxuaxaQwc,My Response,PewDiePie,22,[none],5845909,576597,39774,170708,https://i.ytimg.com/vi/cLdxuaxaQwc/default.jpg,14.496832
3,WYYvHb03Eog,Apple iPhone X first look,The Verge,28,apple iphone x hands on|Apple iPhone X|iPhone ...,2642103,24975,4542,12829,https://i.ytimg.com/vi/WYYvHb03Eog/default.jpg,5.498679
4,sjlHnJvXdQs,iPhone X (parody),jacksfilms,23,jacksfilms|parody|parodies|iphone|iphone x|iph...,1168130,96666,568,6666,https://i.ytimg.com/vi/sjlHnJvXdQs/default.jpg,170.186620
5,cMKX2tE5Luk,The Disaster Artist | Official Trailer HD | A24,A24,1,a24|a24 films|a24 trailers|independent films|t...,1311445,34507,544,3040,https://i.ytimg.com/vi/cMKX2tE5Luk/default.jpg,63.431985
6,8wNr-NQImFg,"The Check In: HUD, Ben Carson and Hurricanes",Late Night with Seth Meyers,23,Late night|Seth Meyers|check in|hud|Ben Carson...,666169,9985,297,1071,https://i.ytimg.com/vi/8wNr-NQImFg/default.jpg,33.619529
7,_HTXMhKWqnA,iPhone X Impressions & Hands On!,Marques Brownlee,28,iPhone X|iphone x|iphone 10|iPhone X impressio...,1728614,74062,2180,15297,https://i.ytimg.com/vi/_HTXMhKWqnA/default.jpg,33.973394
8,_ANP3HR1jsM,ATTACKED BY A POLICE DOG!!,RomanAtwoodVlogs,22,Roman Atwood|Roman|Atwood|roman atwood vlogs|f...,1338533,69687,678,5643,https://i.ytimg.com/vi/_ANP3HR1jsM/default.jpg,102.783186
9,zgLtEob6X-Q,Honest Trailers - The Mummy (2017),Screen Junkies,1,screenjunkies|screen junkies|screenjunkies new...,1056891,29943,878,4046,https://i.ytimg.com/vi/zgLtEob6X-Q/default.jpg,34.103645


## Creating Positive to Negative Comments Ratio

In [15]:
# 1.Calculate the polarity score of an individual comment
from textblob import TextBlob

# Function to calculate sentiment polarity
def get_sentiment(text):
    analysis = TextBlob(text)
    return analysis.sentiment.polarity

# Apply the function to the 'tokenized_text' column
comments['polarity']= comments['tokenized_text'].apply(get_sentiment)

comments.head(10)

,video_id,comment_text,likes,replies,tokenized_text,polarity
0,XpVt6Z1Gjjo,logan paul it s yo big day,4,0,logan paul yo big day,0.00000
1,XpVt6Z1Gjjo,i ve been following you from the start of your...,3,0,following start vine channel seen vlogs,0.00000
2,XpVt6Z1Gjjo,say hi to kong and maverick for me,3,0,say hi kong maverick,0.00000
3,XpVt6Z1Gjjo,my fan attendance,3,0,fan attendance,0.00000
4,XpVt6Z1Gjjo,trending,3,0,trending,0.00000
5,XpVt6Z1Gjjo,on trending ayyeeeee,3,0,trending ayyeeeee,0.00000
6,XpVt6Z1Gjjo,the end though,4,0,end though,0.00000
7,XpVt6Z1Gjjo,trending,3,0,trending,0.00000
8,XpVt6Z1Gjjo,happy one year vlogaversary,3,0,happy one year vlogaversary,0.80000
9,XpVt6Z1Gjjo,you and your shit brother may have single hand...,0,0,shit brother may single handedly ruined youtub...,-0.02381


In [16]:
# 2.Classify the sentiment based on polarity
comments['sentiment_bin'] = comments['polarity'].apply(lambda x: 'positive' if x > 0 else ('negative' if x < 0 else 'neutral'))

In [17]:
comments.head() # Display the first 5 rows of the dataset

,video_id,comment_text,likes,replies,tokenized_text,polarity,sentiment_bin
0,XpVt6Z1Gjjo,logan paul it s yo big day,4,0,logan paul yo big day,0.0,neutral
1,XpVt6Z1Gjjo,i ve been following you from the start of your...,3,0,following start vine channel seen vlogs,0.0,neutral
2,XpVt6Z1Gjjo,say hi to kong and maverick for me,3,0,say hi kong maverick,0.0,neutral
3,XpVt6Z1Gjjo,my fan attendance,3,0,fan attendance,0.0,neutral
4,XpVt6Z1Gjjo,trending,3,0,trending,0.0,neutral


In [18]:
sentiment_df = comments.groupby('video_id')['sentiment_bin'].value_counts().unstack().fillna(0)# Count the number of unique values in the 'sentiment_bin' column

In [19]:
sentiment_df # Display the first 10 rows of the sentiment DataFrame

sentiment_bin,negative,neutral,positive
video_id,,,
--JinobXWPk,18.0,52.0,30.0
-1fzGnFwz9M,19.0,28.0,53.0
-3AGlBYyLjo,2.0,2.0,0.0
-5sCWsLlTCI,17.0,22.0,28.0
-6Zc8Co2H3w,34.0,173.0,193.0
...,...,...,...
zqE-ultsWt0,94.0,195.0,211.0
zrOHeEA14kQ,114.0,249.0,134.0
zuKX0fPlo2Q,0.0,0.0,2.0


In [20]:
# 3.Calculate Positive-to-Negative Ratio
sentiment_df['p_n_comment_ratio'] = sentiment_df['positive'] / sentiment_df['negative']
# Replace inf and -inf with NaN
sentiment_df['p_n_comment_ratio'].replace([np.inf, -np.inf], np.nan, inplace=True)
sentiment_df['p_n_comment_ratio'].fillna(0, inplace=True) # Handle cases where there are no negative comments

C:\Users\New Owner\AppData\Local\Temp\ipykernel_13708\1872454156.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  sentiment_df['p_n_comment_ratio'].replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\New Owner\AppData\Local\Temp\ipykernel_13708\1872454156.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting

In [21]:
# 4.Calculate Percentage of Neutral Comments used in the analysis
# neutral percentage is used to help calculate the emoti0onal charge related to audience reaction.
sentiment_df['neutral_percentage'] = (sentiment_df['neutral'] / (sentiment_df['positive'] + sentiment_df['negative'] + sentiment_df['neutral'])) * 100
sentiment_df['neutral_percentage'] = sentiment_df['neutral_percentage'].round(2) # Round the percentage to 2 decimal places

In [22]:
sentiment_df

sentiment_bin,negative,neutral,positive,p_n_comment_ratio,neutral_percentage
video_id,,,,,
--JinobXWPk,18.0,52.0,30.0,1.666667,52.00
-1fzGnFwz9M,19.0,28.0,53.0,2.789474,28.00
-3AGlBYyLjo,2.0,2.0,0.0,0.000000,50.00
-5sCWsLlTCI,17.0,22.0,28.0,1.647059,32.84
-6Zc8Co2H3w,34.0,173.0,193.0,5.676471,43.25
...,...,...,...,...,...
zqE-ultsWt0,94.0,195.0,211.0,2.244681,39.00
zrOHeEA14kQ,114.0,249.0,134.0,1.175439,50.10
zuKX0fPlo2Q,0.0,0.0,2.0,0.000000,0.00


In [23]:
sentiment_df.head(10)

sentiment_bin,negative,neutral,positive,p_n_comment_ratio,neutral_percentage
video_id,,,,,
--JinobXWPk,18.0,52.0,30.0,1.666667,52.00
-1fzGnFwz9M,19.0,28.0,53.0,2.789474,28.00
-3AGlBYyLjo,2.0,2.0,0.0,0.000000,50.00
-5sCWsLlTCI,17.0,22.0,28.0,1.647059,32.84
-6Zc8Co2H3w,34.0,173.0,193.0,5.676471,43.25
-AJyaVduxCc,44.0,116.0,131.0,2.977273,39.86
-B9z3az6Axc,55.0,207.0,238.0,4.327273,41.40
-C-LJUD2LWU,36.0,49.0,115.0,3.194444,24.50
-CEuQhqNzz4,8.0,161.0,65.0,8.125000,68.80


In [24]:
comments.head(10)

,video_id,comment_text,likes,replies,tokenized_text,polarity,sentiment_bin
0,XpVt6Z1Gjjo,logan paul it s yo big day,4,0,logan paul yo big day,0.00000,neutral
1,XpVt6Z1Gjjo,i ve been following you from the start of your...,3,0,following start vine channel seen vlogs,0.00000,neutral
2,XpVt6Z1Gjjo,say hi to kong and maverick for me,3,0,say hi kong maverick,0.00000,neutral
3,XpVt6Z1Gjjo,my fan attendance,3,0,fan attendance,0.00000,neutral
4,XpVt6Z1Gjjo,trending,3,0,trending,0.00000,neutral
5,XpVt6Z1Gjjo,on trending ayyeeeee,3,0,trending ayyeeeee,0.00000,neutral
6,XpVt6Z1Gjjo,the end though,4,0,end though,0.00000,neutral
7,XpVt6Z1Gjjo,trending,3,0,trending,0.00000,neutral
8,XpVt6Z1Gjjo,happy one year vlogaversary,3,0,happy one year vlogaversary,0.80000,positive
9,XpVt6Z1Gjjo,you and your shit brother may have single hand...,0,0,shit brother may single handedly ruined youtub...,-0.02381,negative


In [25]:
merged_df = sentiment_df.merge(comments[['video_id', 'tokenized_text', 'sentiment_bin']], on='video_id', how='left') # Merge the sentiment DataFrame with the comments DataFrame


In [26]:
merged_df.info() # Display the information about the merged DataFrame

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691374 entries, 0 to 691373
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   video_id            691374 non-null  object 
 1   negative            691374 non-null  float64
 2   neutral             691374 non-null  float64
 3   positive            691374 non-null  float64
 4   p_n_comment_ratio   691374 non-null  float64
 5   neutral_percentage  691374 non-null  float64
 6   tokenized_text      691374 non-null  object 
 7   sentiment_bin       691374 non-null  object 
dtypes: float64(5), object(3)
memory usage: 42.2+ MB


In [27]:
merged_df.head(10) # Display the first 10 rows of the merged DataFrame

,video_id,negative,neutral,positive,p_n_comment_ratio,neutral_percentage,tokenized_text,sentiment_bin
0,--JinobXWPk,18.0,52.0,30.0,1.666667,52.0,saw wandering spider bathroom seriously lying ...,negative
1,--JinobXWPk,18.0,52.0,30.0,1.666667,52.0,stick small ant bee mess spider im petrified,negative
2,--JinobXWPk,18.0,52.0,30.0,1.666667,52.0,brazilian wandering spider deadliest spider,neutral
3,--JinobXWPk,18.0,52.0,30.0,1.666667,52.0,nothing hairspray lighter handle lol,positive
4,--JinobXWPk,18.0,52.0,30.0,1.666667,52.0,snek,neutral
5,--JinobXWPk,18.0,52.0,30.0,1.666667,52.0,hey coyote episode japanese giant hornet unsaf...,positive
6,--JinobXWPk,18.0,52.0,30.0,1.666667,52.0,scorpion fairly easy gauge know large pinchers...,positive
7,--JinobXWPk,18.0,52.0,30.0,1.666667,52.0,wandering sider also family wolf spider wolf s...,positive
8,--JinobXWPk,18.0,52.0,30.0,1.666667,52.0,bday th october im getting book,neutral
9,--JinobXWPk,18.0,52.0,30.0,1.666667,52.0,look like mini dog head,neutral


## Features and Target

In [49]:
X = merged_df['tokenized_text'] # Features
y = merged_df['sentiment_bin'] # Target variable

## Model Training

### TF-IDF & Logistic Regression

In [59]:
vectorizer = TfidfVectorizer() # Initialize the TF-IDF vectorizer
X_vect = vectorizer.fit_transform(X) # Transform the text data into TF-IDF features

In [60]:
X_train_vect, X_test_vect, y_train, y_test = train_test_split(X_vect, y, test_size=0.2, random_state=42) # Split the data into training and testing sets

In [62]:
from sklearn.model_selection import train_test_split # Import train_test_split function
from sklearn.linear_model import LogisticRegression # Import Logistic Regression model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix # Import evaluation metrics

In [64]:
# Encode the target variable
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()  # Initialize the label encoder
y_train_encoded = label_encoder.fit_transform(y_train)  # Encode the training target variable
y_test_encoded = label_encoder.transform(y_test)  # Encode the test target variable

log_model = LogisticRegression(class_weight='balanced')  # Initialize the logistic regression model
log_model.fit(X_train_vect, y_train_encoded)  # Fit the model to the training data
y_pred_encoded = log_model.predict(X_test_vect)  # Make predictions on the test data

# Decode the predictions back to the original labels
y_pred = label_encoder.inverse_transform(y_pred_encoded)  # Decode the predictions

c:\Users\New Owner\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [68]:
report = classification_report(y_test, y_pred) # Generate the classification report
print(report) # Print the classification report

              precision    recall  f1-score   support

    negative       0.93      0.97      0.95     21303
     neutral       0.98      0.99      0.98     58667
    positive       0.99      0.96      0.98     58305

    accuracy                           0.98    138275
   macro avg       0.97      0.97      0.97    138275
weighted avg       0.98      0.98      0.98    138275

